# MiniMax H3 — Optimized ComfyUI Extender on Colab

Optimized for **G4 / RTX PRO 6000 Blackwell 96 GB**. The notebook also detects A100, L4 and T4 and chooses a compatible model profile automatically.

Default G4 path: **FP8 Ref2VA + NVFP4 Qwen3-VL + 4-step Turbo + 0.60 MP + disk-backed Full Batch**.


## 0. Mount Drive + choose persistence

For speed, model weights stay on the Colab VM by default. Output, user state and H3 Extender caches stay on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSIST_MODELS_TO_DRIVE = False   # False = faster model loading from Colab VM disk
PERSIST_OUTPUT_TO_DRIVE = True    # Keep outputs/projects/cache across runtime resets
DRIVE_ROOT = '/content/drive/MyDrive/MiniMax_H3_ComfyUI'


## 1. Check GPU and select the H3 profile


In [ ]:
import subprocess, re, os

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_name = sh("nvidia-smi --query-gpu=name --format=csv,noheader | head -n1")
vram_mb = int(sh("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1"))
vram_gb = vram_mb / 1024

name = gpu_name.lower()
if 'rtx pro 6000' in name or 'blackwell' in name:
    MODEL_PROFILE = 'ref2va-blackwell-fp8'
    H3_VRAM_MODE = 'normalvram'
    H3_RESERVE_VRAM_GB = '6'
elif 'a100' in name:
    MODEL_PROFILE = 'ref2va-a100-int8'
    H3_VRAM_MODE = 'normalvram'
    H3_RESERVE_VRAM_GB = '4'
elif 'l4' in name:
    MODEL_PROFILE = 'ref2va-int8'
    H3_VRAM_MODE = 'normalvram'
    H3_RESERVE_VRAM_GB = '2'
else:
    MODEL_PROFILE = 'ref2va-int8'
    H3_VRAM_MODE = 'lowvram' if vram_gb < 20 else 'normalvram'
    H3_RESERVE_VRAM_GB = '1'

print(f'GPU: {gpu_name} ({vram_gb:.1f} GB)')
print('Model profile:', MODEL_PROFILE)
print('ComfyUI VRAM mode:', H3_VRAM_MODE)
print('Reserved VRAM:', H3_RESERVE_VRAM_GB, 'GB')


## 2. Clone our H3 branch


In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 3. Install / update ComfyUI + latest H3 Extender


In [ ]:
import os
os.environ['COMFY_ROOT'] = '/content/ComfyUI'
os.environ['H3_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['H3_PERSIST_MODELS'] = '1' if PERSIST_MODELS_TO_DRIVE else '0'
os.environ['H3_PERSIST_OUTPUT'] = '1' if PERSIST_OUTPUT_TO_DRIVE else '0'
os.environ['H3_VRAM_MODE'] = H3_VRAM_MODE
os.environ['H3_RESERVE_VRAM_GB'] = H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD'] = 'none'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!bash install_comfy_h3.sh


## 4. Download the GPU-specific H3 Ref2VA models


In [ ]:
MODEL_ROOT = f'{DRIVE_ROOT}/models' if PERSIST_MODELS_TO_DRIVE else '/content/ComfyUI/models'
!python download_models.py --profile "$MODEL_PROFILE" --model-root "$MODEL_ROOT"


## 5. Build the optimized workflows

The main workflow starts at **0.60 MP**, 4 Turbo steps, context 22 and Full Batch disk caching. A Clip-by-Clip fallback is generated automatically.


In [ ]:
!python prepare_workflow.py   --comfy-root /content/ComfyUI   --profile "$MODEL_PROFILE"   --megapixels 0.60   --mode full_batch   --name MiniMax_H3_G4_Optimized_FullBatch.json   --also-safe

!ls -lh /content/ComfyUI/user/default/workflows/


## 6. Launch ComfyUI

Open the printed **Pinggy** URL. The launcher uses memory-safe defaults and disables ComfyUI latent previews during generation to leave more VRAM for H3.


In [ ]:
!bash launch_comfy.sh


## First run

Open **`MiniMax_H3_G4_Optimized_FullBatch.json`**.

For the first G4 test, keep **10-second clips at 0.60 MP**. If the whole sequence completes, try **0.75 MP** next. If a specific project still OOMs, open **`MiniMax_H3_G4_Optimized_Safe_ClipByClip.json`** and generate/validate one clip at a time.
